# Menu Recommender System - Scala (SparkML)

This notebook uses SparkML Linear Regression to recommend menus for each day of the week for people with different allergen restrictions:
- Eggs allergy
- Gluten allergy  
- Lactose intolerance
- Nuts allergy
- Shellfish allergy

In [1]:
// Load Spark dependencies for Scala 2.12.12
// Using Spark 3.5.0 which is compatible with Scala 2.12.12
import $ivy.`org.apache.spark::spark-sql:3.5.0`
import $ivy.`org.apache.spark::spark-mllib:3.5.0`
import $ivy.`org.apache.spark::spark-core:3.5.0`

println("Spark dependencies loaded successfully!")

Spark dependencies loaded successfully!


import $ivy.$                                  

import $ivy.$                                    

import $ivy.$                                   



In [2]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.ml.feature.{VectorAssembler, StringIndexer, OneHotEncoder}
import org.apache.spark.ml.Pipeline
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

// Create Spark Session
val spark = SparkSession.builder()
    .appName("MenuRecommenderScala")
    .master("local[*]")  // Run Spark in local mode with all available cores
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

println("Spark Session created successfully!")

SLF4J: No SLF4J providers were found.
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See https://www.slf4j.org/codes.html#noProviders for further details.


Spark Session created successfully!


import org.apache.spark.sql.SparkSession

import org.apache.spark.ml.regression.LinearRegression

import org.apache.spark.ml.feature.{VectorAssembler, StringIndexer, OneHotEncoder}

import org.apache.spark.ml.Pipeline

import org.apache.spark.sql.functions._

import org.apache.spark.sql.types._

// Create Spark Session

spark: SparkSession = org.apache.spark.sql.SparkSession@3047b3a7

In [3]:
// Load restaurant menus data
val basePath = "output/restaurant_menus"

// Load all restaurant menus
val allMenusDF = spark.read.parquet(s"$basePath/all_restaurant_menus.parquet")

// Load allergen-specific menus
val eggAllergyMenus = spark.read.parquet(s"$basePath/egg_allergy_restaurant_menu.parquet")
val glutenFreeMenus = spark.read.parquet(s"$basePath/gluten_free_restaurant_menu.parquet")
val lactoseIntolerantMenus = spark.read.parquet(s"$basePath/lactose_intolerant_restaurant_menu.parquet")
val nutAllergyMenus = spark.read.parquet(s"$basePath/nut_allergy_restaurant_menu.parquet")
val shellfishAllergyMenus = spark.read.parquet(s"$basePath/shellfish_allergy_restaurant_menu.parquet")

println("Data loaded successfully!")
println(s"Total menus: ${allMenusDF.count()}")
println(s"Egg allergy menus: ${eggAllergyMenus.count()}")
println(s"Gluten-free menus: ${glutenFreeMenus.count()}")
println(s"Lactose intolerant menus: ${lactoseIntolerantMenus.count()}")
println(s"Nut allergy menus: ${nutAllergyMenus.count()}")
println(s"Shellfish allergy menus: ${shellfishAllergyMenus.count()}")

// Show schema
allMenusDF.printSchema()
allMenusDF.show(5, truncate=false)

Data loaded successfully!
Total menus: 99
Egg allergy menus: 20
Gluten-free menus: 20
Lactose intolerant menus: 19
Nut allergy menus: 20
Shellfish allergy menus: 20
root
 |-- meal_id: long (nullable = true)
 |-- starter_title: string (nullable = true)
 |-- starter_ingredients: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- starter_directions: string (nullable = true)
 |-- starter_link: string (nullable = true)
 |-- main_title: string (nullable = true)
 |-- main_ingredients: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_directions: string (nullable = true)
 |-- main_link: string (nullable = true)
 |-- dessert_title: string (nullable = true)
 |-- dessert_ingredients: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- dessert_directions: string (nullable = true)
 |-- dessert_link: string (nullable = true)
 |-- dietary_restriction: string (nullable = true)
 |-- restriction_description: string (nulla

basePath: String = "output/restaurant_menus"
allMenusDF: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
eggAllergyMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
glutenFreeMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
lactoseIntolerantMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
nutAllergyMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]
shellfishAllergyMenus: org.apache.spark.sql.package.DataFrame = [meal_id: bigint, starter_title: string ... 13 more fields]

In [4]:
// Load additional datasets for feature engineering
val nutritionalProfilesDF = spark.read.parquet("output/nutritional_profiles/nutritional_profiles.parquet")
val recipesDF = spark.read.parquet("data/recipes_data.parquet")

println("Additional datasets loaded!")
nutritionalProfilesDF.printSchema()

Additional datasets loaded!
root
 |-- fdc_id: long (nullable = true)
 |-- food_description: string (nullable = true)
 |-- food_type: string (nullable = true)
 |-- total_nutrients: long (nullable = true)
 |-- energy: double (nullable = true)
 |-- protein: double (nullable = true)
 |-- carbs: double (nullable = true)
 |-- total_fat: double (nullable = true)
 |-- water: double (nullable = true)
 |-- ash: double (nullable = true)
 |-- alcohol: double (nullable = true)
 |-- caffeine: double (nullable = true)
 |-- fiber: double (nullable = true)
 |-- sugars: double (nullable = true)
 |-- glucose: double (nullable = true)
 |-- fructose: double (nullable = true)
 |-- sucrose: double (nullable = true)
 |-- lactose: double (nullable = true)
 |-- saturated_fat: double (nullable = true)
 |-- monounsaturated_fat: double (nullable = true)
 |-- polyunsaturated_fat: double (nullable = true)
 |-- trans_fat: double (nullable = true)
 |-- cholesterol: double (nullable = true)
 |-- vitamin_a: double (null

nutritionalProfilesDF: org.apache.spark.sql.package.DataFrame = [fdc_id: bigint, food_description: string ... 46 more fields]
recipesDF: org.apache.spark.sql.package.DataFrame = [title: string, ingredients: string ... 5 more fields]

In [5]:
// Define allergen categories
val allergenCategories = Map(
    "eggs" -> eggAllergyMenus,
    "gluten" -> glutenFreeMenus,
    "lactose" -> lactoseIntolerantMenus,
    "nuts" -> nutAllergyMenus,
    "shellfish" -> shellfishAllergyMenus
)

// Days of the week
val daysOfWeek = Array("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday")

println(s"Allergen categories: ${allergenCategories.keys.mkString(", ")}")
println(s"Days of week: ${daysOfWeek.mkString(", ")}")

Allergen categories: lactose, eggs, gluten, nuts, shellfish
Days of week: Monday, Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday


allergenCategories: Map[String, org.apache.spark.sql.package.DataFrame] = Map(
  "lactose" -> [meal_id: bigint, starter_title: string ... 13 more fields],
  "eggs" -> [meal_id: bigint, starter_title: string ... 13 more fields],
  "gluten" -> [meal_id: bigint, starter_title: string ... 13 more fields],
  "nuts" -> [meal_id: bigint, starter_title: string ... 13 more fields],
  "shellfish" -> [meal_id: bigint, starter_title: string ... 13 more fields]
)
daysOfWeek: Array[String] = Array(
  "Monday",
  "Tuesday",
  "Wednesday",
  "Thursday",
  "Friday",
  "Saturday",
  "Sunday"
)

In [6]:
// Function to prepare features for linear regression
// NOTE: If you see errors about 'rating' column, restart the kernel and re-run all cells
def prepareFeatures(df: org.apache.spark.sql.DataFrame): org.apache.spark.sql.DataFrame = {
    // Add day of week feature (if not present)
    val dfWithDay = if (df.columns.contains("day_of_week")) {
        df
    } else {
        df.withColumn("day_of_week", 
            when(rand() < 0.143, "Monday")
            .when(rand() < 0.286, "Tuesday")
            .when(rand() < 0.429, "Wednesday")
            .when(rand() < 0.572, "Thursday")
            .when(rand() < 0.715, "Friday")
            .when(rand() < 0.858, "Saturday")
            .otherwise("Sunday"))
    }
    
    // Create a preference score (target variable for regression)
    // This could be based on ratings, nutritional value, or other metrics
    val dfWithScore = if (df.columns.contains("preference_score")) {
        dfWithDay
    } else {
        // Create a synthetic preference score based on available features
        // Since 'rating' column doesn't exist, use a base score with randomness
        // You can modify this to use actual features from your data (e.g., meal_id, ingredients count, etc.)
        dfWithDay.withColumn("preference_score", 
            lit(3.5) + rand() * 2.0 - 1.0) // Base score of 3.5 with randomness between 2.5 and 4.5
    }
    
    dfWithScore
}

defined function prepareFeatures

In [7]:
// Function to train linear regression model and generate recommendations
def trainModelAndRecommend(
    allergenType: String,
    menuDF: org.apache.spark.sql.DataFrame,
    dayOfWeek: String
): org.apache.spark.sql.DataFrame = {
    
    println(s"\n=== Processing: $allergenType - $dayOfWeek ===")
    
    // Prepare features
    val preparedDF = prepareFeatures(menuDF)
    
    // Filter for the specific day if day_of_week column exists
    val dayDF = if (preparedDF.columns.contains("day_of_week")) {
        preparedDF.filter(col("day_of_week") === dayOfWeek)
    } else {
        preparedDF // Use all data if day filtering not available
    }
    
    if (dayDF.count() == 0) {
        println(s"No data available for $allergenType on $dayOfWeek")
        return spark.emptyDataFrame
    }
    
    // Identify feature columns (exclude target and metadata columns)
    val excludeCols = Array("preference_score", "day_of_week", "menu_id", "restaurant_id", "menu_name", "restaurant_name")
    val featureCols = dayDF.columns.filterNot(excludeCols.contains)
    
    // Select numeric columns for features only (VectorAssembler doesn't support StringType)
    val numericCols = featureCols.filter { colName =>
        val dtype = dayDF.schema(colName).dataType
        dtype.isInstanceOf[NumericType]  // Only numeric types, exclude StringType
    }
    
    if (numericCols.length == 0) {
        println(s"No suitable feature columns found for $allergenType")
        return dayDF.limit(10) // Return top 10 as fallback
    }
    
    // Create feature vector
    val assembler = new VectorAssembler()
        .setInputCols(numericCols.take(10)) // Limit to first 10 numeric columns
        .setOutputCol("features")
        .setHandleInvalid("skip")
    
    // Select columns properly - convert all to Column objects
    // Exclude preference_score from dayDF.columns since we're adding it explicitly
    val otherCols = dayDF.columns.filterNot(_ == "preference_score").map(col)
    val allCols = Seq(col("features"), col("preference_score")) ++ otherCols
    val featureDF = assembler.transform(dayDF)
        .select(allCols: _*)
        .filter(col("features").isNotNull)
    
    if (featureDF.count() == 0) {
        println(s"No valid features for $allergenType on $dayOfWeek")
        return dayDF.limit(10)
    }
    
    // Train Linear Regression model
    val lr = new LinearRegression()
        .setLabelCol("preference_score")
        .setFeaturesCol("features")
        .setMaxIter(10)
        .setRegParam(0.3)
        .setElasticNetParam(0.8)
    
    val model = lr.fit(featureDF)
    
    // Make predictions
    val predictions = model.transform(featureDF)
        .withColumn("allergen_type", lit(allergenType))
        .withColumn("recommended_day", lit(dayOfWeek))
    
    // Get top recommendations (highest predicted scores)
    // Build column list properly - convert all to Column objects
    val recOtherCols = dayDF.columns.filterNot(Array("features", "preference_score").contains).map(col)
    val recCols = Seq(col("allergen_type"), col("recommended_day"), col("prediction")) ++ recOtherCols
    val recommendations = predictions
        .orderBy(desc("prediction"))
        .limit(5)
        .select(recCols: _*)
    
    println(s"Generated ${recommendations.count()} recommendations for $allergenType on $dayOfWeek")
    
    recommendations
}

defined function trainModelAndRecommend

In [8]:
// Generate recommendations for all allergen categories and days
import scala.collection.mutable.ListBuffer

val allRecommendations = ListBuffer[org.apache.spark.sql.DataFrame]()

for ((allergenType, menuDF) <- allergenCategories) {
    for (day <- daysOfWeek) {
        val recommendations = trainModelAndRecommend(allergenType, menuDF, day)
        if (recommendations.count() > 0) {
            allRecommendations += recommendations
        }
    }
}

// Combine all recommendations
val finalRecommendations = if (allRecommendations.nonEmpty) {
    allRecommendations.reduce(_ union _)
} else {
    spark.emptyDataFrame
}

println(s"\n=== Total Recommendations Generated: ${finalRecommendations.count()} ===")


=== Processing: lactose - Monday ===
Generated 5 recommendations for lactose on Monday

=== Processing: lactose - Tuesday ===
Generated 3 recommendations for lactose on Tuesday

=== Processing: lactose - Wednesday ===
Generated 5 recommendations for lactose on Wednesday

=== Processing: lactose - Thursday ===
Generated 3 recommendations for lactose on Thursday

=== Processing: lactose - Friday ===
No data available for lactose on Friday

=== Processing: lactose - Saturday ===
No data available for lactose on Saturday

=== Processing: lactose - Sunday ===
No data available for lactose on Sunday

=== Processing: eggs - Monday ===
Generated 1 recommendations for eggs on Monday

=== Processing: eggs - Tuesday ===
Generated 5 recommendations for eggs on Tuesday

=== Processing: eggs - Wednesday ===
Generated 3 recommendations for eggs on Wednesday

=== Processing: eggs - Thursday ===
Generated 3 recommendations for eggs on Thursday

=== Processing: eggs - Friday ===
Generated 3 recommendat

import scala.collection.mutable.ListBuffer


allRecommendations: ListBuffer[org.apache.spark.sql.package.DataFrame] = ListBuffer(
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: string, recommended_day: string ... 17 more fields],
  [allergen_type: 

In [9]:
// Display recommendations grouped by allergen type and day
for ((allergenType, _) <- allergenCategories) {
    println(s"\n{'='*60}")
    println(s"RECOMMENDATIONS FOR ${allergenType.toUpperCase} ALLERGY")
    println(s"{'='*60}")
    
    for (day <- daysOfWeek) {
        val dayRecs = finalRecommendations
            .filter(col("allergen_type") === allergenType && col("recommended_day") === day)
            .orderBy(desc("prediction"))
        
        println(s"\n--- $day ---")
        dayRecs.show(5, truncate=false)
    }
}


{'='*60}
RECOMMENDATIONS FOR LACTOSE ALLERGY
{'='*60}

--- Monday ---
+-------------+---------------+-----------------+-------+----------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [10]:
// Save recommendations to parquet
val outputPath = "output/menu_recommendations_scala"
finalRecommendations
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .parquet(outputPath)

println(s"Recommendations saved to: $outputPath")

// Convert array columns to strings for CSV export (CSV doesn't support ARRAY types)
val csvDF = finalRecommendations.columns.foldLeft(finalRecommendations) { (df, colName) =>
    val dtype = df.schema(colName).dataType
    if (dtype.isInstanceOf[ArrayType]) {
        // Convert array to comma-separated string
        df.withColumn(colName, concat_ws(", ", col(colName)))
    } else {
        df
    }
}

// Save as CSV for easier viewing
csvDF
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(s"$outputPath/csv")

println(s"CSV version saved to: $outputPath/csv")

Recommendations saved to: output/menu_recommendations_scala
CSV version saved to: output/menu_recommendations_scala/csv


outputPath: String = "output/menu_recommendations_scala"
csvDF: org.apache.spark.sql.package.DataFrame = [allergen_type: string, recommended_day: string ... 17 more fields]

In [11]:
// Summary statistics
val summaryDF = finalRecommendations
    .groupBy("allergen_type", "recommended_day")
    .agg(
        count("*").alias("num_recommendations"),
        avg("prediction").alias("avg_prediction_score"),
        max("prediction").alias("max_prediction_score"),
        min("prediction").alias("min_prediction_score")
    )
    .orderBy("allergen_type", "recommended_day")

println("\n=== Recommendation Summary Statistics ===")
summaryDF.show(35, truncate=false)


=== Recommendation Summary Statistics ===
+-------------+---------------+-------------------+--------------------+--------------------+--------------------+
|allergen_type|recommended_day|num_recommendations|avg_prediction_score|max_prediction_score|min_prediction_score|
+-------------+---------------+-------------------+--------------------+--------------------+--------------------+
|eggs         |Friday         |3                  |3.159736705914368   |3.441340142934561   |2.8946981769541864  |
|eggs         |Monday         |1                  |2.812214221289248   |2.812214221289248   |2.812214221289248   |
|eggs         |Saturday       |2                  |3.205059351740507   |3.2490496636112853  |3.1610690398697283  |
|eggs         |Thursday       |3                  |3.355865046047189   |3.355865046047189   |3.355865046047189   |
|eggs         |Tuesday        |5                  |3.319110127266697   |3.319110127266697   |3.319110127266697   |
|eggs         |Wednesday      |3     

summaryDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [allergen_type: string, recommended_day: string ... 4 more fields]

In [12]:
// Cleanup
spark.stop()
println("Spark session stopped.")

Spark session stopped.
